In [6]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
from pathlib import Path
import xcdat as xc
import cmasher as cmr
import xarray as xr
import sys

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *

# Reload edited Python modules automatically before running subsequent cells.
%load_ext autoreload
%autoreload 2

import sys

# Add the project folder only once, even if this cell is rerun.
functions_path = "/scratch/leiff/MCA/amip_clean/project_functions/"
if functions_path not in sys.path:
    sys.path.append(functions_path)

from MCA_cov import *
from significance import *
from plotting import *
from diagnostics import *
# from taylor_setup import *
from setup_diagnostic import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# 1. Settings and imports
from pathlib import Path
from datetime import datetime, timezone
import json, re
import numpy as np
import xarray as xr

historical_root = Path('/scratch/leiff/MCA/amip_clean/data/HIST')
output_directory = Path.cwd()
first_month, last_month = '2000-03', '2014-12'

# Turn calendar year/month into an integer so different model calendars can be
# compared without pretending that their exact day timestamps are identical.
first_year, first_number = map(int, first_month.split('-'))
last_year, last_number = map(int, last_month.split('-'))
wanted_months = np.arange(12 * first_year + first_number - 1,
                         12 * last_year + last_number)
member_pattern = re.compile(r'r\d+i\d+p\d+f\d+')
expected_lat = np.arange(-88.75, 90., 2.5)
expected_lon = np.arange(1.25, 360., 2.5)

if not historical_root.is_dir():
    raise FileNotFoundError(f'Historical directory not found; edit historical_root: {historical_root}')
print(f'Inspecting: {historical_root}')



Inspecting: /scratch/leiff/MCA/amip_clean/data/HIST


In [8]:
# 2. Read headers, month coverage, grid coordinates and explicit member labels
# Each record represents one physical file. Include both native and remapped
# files if present; the grid check identifies exact g025 candidates. Never
# invent member labels from an ensemble count, or infer p2 from a filename.
files = sorted(path for path in historical_root.rglob('*.nc')
               if path.name.startswith(('tas_', 'N_')))
if not files:
    raise FileNotFoundError('No tas_*.nc or N_*.nc files found. Check the directory and filename convention.')
records = []
for index, path in enumerate(files, start=1):
    record = {'path': str(path), 'filename_model_hint': path.parent.name}
    try:
        record['file_bytes'] = path.stat().st_size
        # Loading dimension coordinates is small compared with a climate field.
        # No data-variable .values/.load() call occurs in this inventory.
        with xr.open_dataset(path, chunks=None, cache=False) as ds:
            record['dimensions'] = dict(ds.sizes)
            record['global_attributes'] = dict(ds.attrs)
            record['variables'] = {name: {'dimensions': list(var.dims),
                'dtype': str(var.dtype), 'attributes': dict(var.attrs)}
                for name, var in ds.data_vars.items()}
            coordinate_names = [name for name, var in ds.variables.items()
                                if var.ndim <= 1 and var.size <= 10000
                                and (name in ds.coords or name in ds.dims)]
            record['coordinates'] = {name: ds[name].values.tolist()
                for name in coordinate_names if name.lower() not in ('time', 'year_month')}

            # Preserve all explicit labels, including any p1 CanESM5 labels.
            # Selection/reuse will enforce p2 later; this cell changes no files.
            labels = set()
            for name in coordinate_names:
                values = np.asarray(ds[name].values)
                if values.dtype.kind in ('U', 'S', 'O') and name.lower() != 'time':
                    for value in values.ravel():
                        text = value.decode() if isinstance(value, bytes) else str(value)
                        if member_pattern.fullmatch(text):
                            labels.add(text)
            if member_pattern.fullmatch(str(ds.attrs.get('variant_label', ''))):
                labels.add(str(ds.attrs['variant_label']))
            record['explicit_member_labels'] = sorted(labels)
            record['member_labels_missing'] = not bool(labels)
            model_name = str(ds.attrs.get('source_id', path.parent.name))
            if 'CanESM5' in model_name or 'CanESM5' in path.name:
                record['CanESM5_p2_labels'] = sorted(label for label in labels
                                                    if re.search(r'i\d+p2f', label))
                record['CanESM5_other_physics_labels'] = sorted(
                    labels - set(record['CanESM5_p2_labels']))

            if 'time' in ds.coords:
                time = ds.time
                months = np.asarray(time.dt.year * 12 + time.dt.month - 1, dtype=int)
                unique_months, month_counts = np.unique(months, return_counts=True)
                missing = np.setdiff1d(wanted_months, unique_months)
                duplicates = unique_months[(month_counts > 1) & np.isin(unique_months, wanted_months)]
                record['time'] = {'count': int(time.size),
                    'first': str(time.values[0]) if time.size else None,
                    'last': str(time.values[-1]) if time.size else None,
                    'calendar': time.encoding.get('calendar', time.attrs.get('calendar')),
                    'strictly_increasing_months': bool(np.all(np.diff(months) > 0)),
                    'missing_overlap_month_ids': missing.tolist(),
                    'duplicate_overlap_month_ids': duplicates.tolist(),
                    'complete_overlap': bool(not missing.size and not duplicates.size)}
            else:
                record['time'] = {'complete_overlap': False, 'problem': 'No time coordinate found'}

            lat_name = next((name for name in ('lat', 'latitude') if name in ds.coords), None)
            lon_name = next((name for name in ('lon', 'longitude') if name in ds.coords), None)
            record['exact_reference_g025'] = False
            if lat_name and lon_name:
                lat, lon = np.asarray(ds[lat_name]), np.asarray(ds[lon_name])
                record['exact_reference_g025'] = bool(
                    lat.shape == expected_lat.shape and lon.shape == expected_lon.shape
                    and np.allclose(lat, expected_lat, atol=1e-8, rtol=0)
                    and np.allclose(lon, expected_lon, atol=1e-8, rtol=0))
        record['status'] = 'header_read'
    except Exception as error:
        # A broken/unreadable file stays visible while the remaining files are
        # inspected. This is a report of errors, not a silent population filter.
        record.update(status='error', error=f'{type(error).__name__}: {error}')
    records.append(record)
    if index % 20 == 0 or index == len(files):
        print(f'Inspected {index}/{len(files)} files')



Inspected 20/106 files
Inspected 40/106 files
Inspected 60/106 files
Inspected 80/106 files
Inspected 100/106 files
Inspected 106/106 files


In [9]:
# 3. Save the small report to return to the Covariance chat
# Convert unusual NetCDF attribute types to printable text. This does not alter
# the source metadata and does not serialize any tas or N field payloads.
timestamp = datetime.now(timezone.utc)
report = {'created_utc': timestamp.isoformat(), 'historical_root': str(historical_root),
    'requested_period': [first_month, last_month], 'requested_month_count': len(wanted_months),
    'scope': 'Headers and small coordinates only; no climate-field payload validation',
    'files': records}
output_directory.mkdir(parents=True, exist_ok=True)
report_path = output_directory / f'historical_reuse_inventory_{timestamp:%Y%m%dT%H%M%S%fZ}.json'
report_path.write_text(json.dumps(report, indent=2, default=str) + '\n')
print(f'\nSaved: {report_path}')
print(f'Files: {len(records)}; header errors: {sum(row["status"] == "error" for row in records)}')
print('Return this JSON report; the large NetCDF files can stay on the HPC.')



Saved: /scratch/leiff/MCA/amip_clean/historical_reuse_inventory_20261027T175343845171Z.json
Files: 106; header errors: 0
Return this JSON report; the large NetCDF files can stay on the HPC.
